# 05 · Temporal kNN on the d8 residual — relevance / analog probes

*Edge-features arc · 01 discovery · 02 linear base · 03 hero cascade · 04 regime-MoE · 05 temporal-kNN  —  machinery: `ridge_pipeline_throughline.ipynb`*

**Verify first — nothing pasted.** A model-free probe of the *same* d8@cs0.5 leftover the MoE/EBM model: predict the
h16-19 residual from **regime-similar past close bars** (self-attention ≈ learned kNN; here a fixed-metric causal
kNN). The Hero-A base QLIKE below is **recomputed from the saved leftover preds** (`preds/fa_d8c5.csv`) with the
real `apply_duan_smearing` metric and asserted to 0.12081; the two leakage-guarded variants (`knn_d8.sbatch`) are
read from `knn_d8.csv`, and each carries a **shuffle placebo** (shuffle the neighbour residuals -> must give ~0;
a gain there = look-ahead artifact) which the Verify cell asserts is `<5e-4`.

In [1]:
import html, inspect, json, os, sys, textwrap
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

def find_repo(s):
    for q in [Path(s).resolve(), *Path(s).resolve().parents]:
        if (q / "resid_amortized.py").exists() and (q / "src").is_dir():
            return q
    raise FileNotFoundError("repo root")
REPO = find_repo(Path.cwd()); os.chdir(REPO); sys.path.insert(0, str(REPO))
LADDER = REPO / "results" / "moe_ladder"

# Collapsible, theme-following source display (one <details> per object; folded).
def _details(f, open_=False):
    mod = f.__module__.replace("src.", "src/").replace(".", "/") + ".py"
    try:
        sig = ("class " + f.__name__) if inspect.isclass(f) else ("def " + f.__name__ + str(inspect.signature(f)))
    except (ValueError, TypeError):
        sig = f.__qualname__
    body = "```python\n" + textwrap.dedent(inspect.getsource(f)).rstrip() + "\n```"
    return (f"<details{' open' if open_ else ''}>\n<summary><code>{html.escape(mod + '  ·  ' + sig)}"
            f"</code></summary>\n\n{body}\n\n</details>")
def show_one(f):
    return Markdown(_details(f))
def show_src(path, lang="python", open_=False):  # fold a whole source FILE (scripts have no importable fn)
    p = Path(path); txt = p.read_text(encoding="utf-8").rstrip()
    return Markdown(f"<details{' open' if open_ else ''}>\n<summary><code>{html.escape(str(p))}"
                    f"</code></summary>\n\n```{lang}\n{txt}\n```\n\n</details>")

from src.evaluation.metrics import apply_duan_smearing  # the real metric (folded in s1; recomputed, never pasted)

def qlike(pred_adj, y_true, base):
    """Exact QLIKE the kNN scripts report: Duan-smear to raw scale, then mean(r - log r - 1)."""
    pr, tr = apply_duan_smearing(np.asarray(pred_adj, np.float64),
                                 np.asarray(y_true, np.float64), np.asarray(base, np.float64))
    m = (tr > 0) & (pr > 0); r = tr[m] / pr[m]
    return float(np.mean(r - np.log(r) - 1.0))
print("setup ok")

setup ok


---
## 1 . The two variants + the metric (folded)

- **`knn_analog`** -- causal kNN analog: weighted **mean** of the K nearest past close analogs, strict
  embargo >= HAR max lag (3125 bars) so neighbour residuals are fully realised. Carries a shuffle placebo.
- **`knn_local`** -- large-K **local-linear ridge** relevance regression (the "proper" variant; naive
  K-mean over-adds variance and hurts). Prior standalone: a sliver on the close leftover, shuffle-clean,
  but **dominated by the EBM regime**. Re-run here on the linbest `fa_d8c5` base for an apples-to-apples row.

The **leakage guards** live in the folded source: the embargo loop (`thr = kc[i] - EMBARGO`, `EMBARGO = 3125`)
and the SHUFFLE placebo (`rng.permutation` over the neighbour residuals). `apply_duan_smearing` is the metric
the Verify cell recomputes the base QLIKE with.

In [2]:
import knn_analog, knn_local
display(show_one(knn_analog.main))      # causal kNN analog -- embargo>=3125 + shuffle placebo
display(show_one(knn_local.main))       # large-K local-linear ridge relevance + shuffle placebo
display(show_one(apply_duan_smearing))  # the metric the Verify cell recomputes with (no pasted numbers)

<details>
<summary><code>knn_analog.py  ·  def main() -&gt; None</code></summary>

```python
def main() -> None:
    d = f"{CACHE_ROOT}/{CELL}"
    feats = json.load(open(f"{d}/feats.json"))
    Xs = np.load(f"{d}/Xs.npy", mmap_mode="r")

    parts = [
        pd.read_csv(f) for f in glob.glob(f"results/resid_ab/{CELL}/{LBL}/chunk_*.csv")
    ]
    A = pd.concat(parts, ignore_index=True).sort_values("k")
    k = A["k"].to_numpy().astype(np.int64)
    pred_adj = A["pred_adj"].to_numpy(np.float64)
    y_true = A["y_true"].to_numpy(np.float64)
    base = A["base"].to_numpy(np.float64)
    t = k + TRAIN_WIN

    hour = np.asarray(Xs[t, feats.index("hour")], dtype=np.float64)
    E = np.column_stack(
        [np.asarray(Xs[t, feats.index(c)], dtype=np.float64) for c in EMB_COLS] + [hour]
    )
    E = (E - E.mean(0)) / (E.std(0) + 1e-9)
    r = y_true - pred_adj
    close = (hour >= 16) & (hour <= 19)
    ci = np.where(close)[0]
    kc, Ec, rc = k[ci], np.ascontiguousarray(E[ci]), r[ci]
    nC = len(ci)
    print(
        "OOS rows=%d close bars=%d  embargo=%d  emb_dim=%d"
        % (len(k), nC, EMBARGO, E.shape[1]),
        flush=True,
    )

    def anhat(rsrc, K):
        ah = np.zeros(nC, dtype=np.float64)
        ptr = 0  # eligible = close bars [0:ptr) with kc <= kc[i]-EMBARGO (kc ascending)
        for i in range(nC):
            thr = kc[i] - EMBARGO
            while ptr < nC and kc[ptr] <= thr:
                ptr += 1
            if ptr < K:
                continue
            dd = Ec[:ptr] - Ec[i]
            dist = np.einsum("ij,ij->i", dd, dd)
            nn = np.argpartition(dist, K)[:K]
            ah[i] = rsrc[nn].mean()
        return ah

    q0 = qlike(pred_adj, y_true, base)
    q0c = qlike(pred_adj[ci], y_true[ci], base[ci])
    print("Hero A: full=%.5f  h16-19=%.5f" % (q0, q0c), flush=True)
    rng = np.random.RandomState(0)
    for K in (25, 50, 100):
        ah = anhat(rc, K)
        new = pred_adj.copy()
        new[ci] += ah
        q1 = qlike(new, y_true, base)
        q1c = qlike(new[ci], y_true[ci], base[ci])
        ahs = anhat(rc[rng.permutation(nC)], K)
        news = pred_adj.copy()
        news[ci] += ahs
        qs = qlike(news, y_true, base)
        print(
            "K=%3d full %.5f (d%+.6f) | SHUFFLE %.5f (d%+.6f) | h16-19 %.5f (d%+.6f)  ah|mean|=%.2e"
            % (K, q1, q1 - q0, qs, qs - q0, q1c, q1c - q0c, np.abs(ah[ah != 0]).mean()),
            flush=True,
        )
    print("KNN_ANALOG_DONE", flush=True)
```

</details>

<details>
<summary><code>knn_local.py  ·  def main() -&gt; None</code></summary>

```python
def main() -> None:
    d = f"{CACHE_ROOT}/{CELL}"
    feats = json.load(open(f"{d}/feats.json"))
    Xs = np.load(f"{d}/Xs.npy", mmap_mode="r")
    A = pd.concat(
        [
            pd.read_csv(f)
            for f in glob.glob(f"results/resid_ab/{CELL}/{LBL}/chunk_*.csv")
        ]
    ).sort_values("k")
    k = A["k"].to_numpy().astype(np.int64)
    pred_adj, y_true, base = (
        A[c].to_numpy(np.float64) for c in ("pred_adj", "y_true", "base")
    )
    r = y_true - pred_adj
    t = k + TRAIN_WIN
    hour = np.asarray(Xs[t, feats.index("hour")], dtype=np.float64)
    close = (hour >= 16) & (hour <= 19)
    ci = np.where(close)[0]
    kc, rc = k[ci], r[ci]
    tc = t[ci]

    def grab(cols):
        return (
            np.ascontiguousarray(
                np.column_stack(
                    [np.asarray(Xs[tc, feats.index(c)], dtype=np.float64) for c in cols]
                )
            )
            if cols
            else np.zeros((len(ci), 0))
        )

    Esim = grab(SIM)
    Esim = (Esim - Esim.mean(0)) / (Esim.std(0) + 1e-9)
    Fmats = {name: grab(cols) for name, cols in FSETS.items()}
    nC = len(ci)
    print(
        "close=%d  K=%d  embargo=%d  resid_std=%.4f" % (nC, K, EMBARGO, rc.std()),
        flush=True,
    )

    preds = {name: np.zeros(nC) for name in FSETS}
    rng = np.random.RandomState(0)
    rc_shuf = rc[rng.permutation(nC)]
    preds_shuf_all = np.zeros(nC)  # 'all' fset on shuffled residuals (placebo)
    ptr = 0
    for i in range(nC):
        thr = kc[i] - EMBARGO
        while ptr < nC and kc[ptr] <= thr:
            ptr += 1
        if ptr <= K:
            continue
        dd = Esim[:ptr] - Esim[i]
        dist = np.einsum("ij,ij->i", dd, dd)
        nn = np.argpartition(dist, K)[:K]
        for name, F in Fmats.items():
            Dtr = np.column_stack([np.ones(K), F[nn]])
            G = Dtr.T @ Dtr + RIDGE * np.eye(Dtr.shape[1])
            beta = np.linalg.solve(G, Dtr.T @ rc[nn])
            preds[name][i] = np.concatenate([[1.0], F[i]]) @ beta
        Da = np.column_stack([np.ones(K), Fmats["all"][nn]])
        Ga = Da.T @ Da + RIDGE * np.eye(Da.shape[1])
        bshuf = np.linalg.solve(Ga, Da.T @ rc_shuf[nn])
        preds_shuf_all[i] = np.concatenate([[1.0], Fmats["all"][i]]) @ bshuf

    q0 = qlike(pred_adj, y_true, base)
    q0c = qlike(pred_adj[ci], y_true[ci], base[ci])
    print("Hero A: full=%.5f  h16-19=%.5f" % (q0, q0c), flush=True)
    for name in FSETS:
        new = pred_adj.copy()
        new[ci] += preds[name]
        print(
            "  %-9s full %.5f (d%+.6f)  h16-19 %.5f (d%+.6f)"
            % (
                name,
                qlike(new, y_true, base),
                qlike(new, y_true, base) - q0,
                qlike(new[ci], y_true[ci], base[ci]),
                qlike(new[ci], y_true[ci], base[ci]) - q0c,
            ),
            flush=True,
        )
    new = pred_adj.copy()
    new[ci] += preds_shuf_all
    print(
        "  %-9s full %.5f (d%+.6f)  [should be ~0]"
        % ("SHUFFLE", qlike(new, y_true, base), qlike(new, y_true, base) - q0),
        flush=True,
    )
    print("KNN_LOCAL_DONE", flush=True)
```

</details>

<details>
<summary><code>src/evaluation/metrics.py  ·  def apply_duan_smearing(forecasts: &#x27;np.ndarray&#x27;, y_true: &#x27;np.ndarray&#x27;, baselines: &#x27;np.ndarray&#x27;) -&gt; &#x27;tuple[np.ndarray, np.ndarray]&#x27;</code></summary>

```python
def apply_duan_smearing(
    forecasts: np.ndarray,
    y_true: np.ndarray,
    baselines: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Apply Duan smearing correction to convert adjusted-scale forecasts to raw scale.

    Parameters
    ----------
    forecasts : array-like
        Model predictions on adjusted (sqrt / log) scale.
    y_true : array-like
        True values on adjusted scale.
    baselines : array-like
        Baseline volatility used to scale back to raw units.

    Returns
    -------
    pred_raw : np.ndarray
        Smearing-corrected predictions on raw scale.
    true_raw : np.ndarray
        True values on raw scale.
    """
    forecasts = np.asarray(forecasts, dtype=np.float64)
    y_true = np.asarray(y_true, dtype=np.float64)
    baselines = np.asarray(baselines, dtype=np.float64)

    smear = np.mean((y_true - forecasts) ** 2)
    pred_raw = (forecasts**2 + smear) * baselines
    true_raw = (y_true**2) * baselines
    return pred_raw, true_raw
```

</details>

---
## 2 . Verify -- recompute the Hero-A base, then the kNN deltas

(a) The Hero-A base QLIKE is **recomputed** from the saved leftover preds `preds/fa_d8c5.csv`
(cols `k, pred_adj, y_true, base`) via `qlike(...)` above -- asserted to reproduce **0.12081**.
(b) The kNN deltas are read from `knn_d8.csv` (parsed from the `knn_d8.sbatch` run): the naive analog
should **hurt** (variance), the proper local-ridge should **help a sliver**, and the **SHUFFLE** placebo
`|d_full|` must be `<5e-4` (leakage-clean). Reproduce-cmd: `bash knn_d8.sbatch`.

In [3]:
# (a) RECOMPUTE the Hero-A base QLIKE from the saved leftover preds (not pasted).
fp = LADDER / "preds" / "fa_d8c5.csv"
if fp.exists() and fp.stat().st_size > 0:
    fa = pd.read_csv(fp)  # cols: k, pred_adj, y_true, base
    q_base = qlike(fa.pred_adj, fa.y_true, fa.base)
    print(f"Hero-A base (fa_d8c5) recomputed QLIKE = {q_base:.5f}  (n={len(fa)})")
    assert round(q_base, 5) == 0.12081, f"Hero-A base did not reproduce 0.12081 (got {q_base:.5f})"
    print("PASS - Hero-A base reproduces 0.12081 from saved preds.")
else:
    print("PENDING - preds/fa_d8c5.csv absent. Reproduce: bash knn_d8.sbatch "
          "(writes the fa_d8c5 leftover chunks, collected here).")

# (b) The two kNN variants + the SHUFFLE placebo (parsed from the knn_d8 run).
kp = LADDER / "knn_d8.csv"
if kp.exists() and kp.stat().st_size > 0:
    knn = pd.read_csv(kp); display(knn)
    base_row = knn[knn.variant == "hero_a"]
    assert round(float(base_row.qlike_full.iloc[0]), 5) == 0.12081, "table base disagrees with the recompute"
    shuf = knn[knn.fset.astype(str).str.contains("SHUFFLE", case=False, na=False)]
    assert len(shuf) and (shuf.d_full.abs() < 5e-4).all(), "SHUFFLE placebo not ~0 -- look-ahead artifact!"
    analog = knn[knn.variant == "knn_analog"]
    local = knn[(knn.variant == "knn_local") & ~knn.fset.astype(str).str.contains("SHUFFLE", case=False, na=False)]
    assert (analog.d_full > 0).all(), "naive analog should HURT (variance) on this low-dim leftover"
    assert local.d_full.min() < 0, "proper local-ridge should help a sliver"
    print(f"PASS - SHUFFLE |d_full|={shuf.d_full.abs().max():.6f} < 5e-4 (leakage-clean); "
          f"analog hurts (+{analog.d_full.max():.5f}), local-ridge best d_full={local.d_full.min():+.5f}.")
else:
    print("PENDING - knn_d8.csv absent. Reproduce: bash knn_d8.sbatch "
          "(or: KNN_CELL=... KNN_LBL=resid_subset_fa_d8c5 $PY knn_analog.py ; $PY knn_local.py).")

Hero-A base (fa_d8c5) recomputed QLIKE = 0.12081  (n=194934)
PASS - Hero-A base reproduces 0.12081 from saved preds.


,variant,fset,qlike_full,d_full,qlike_h16_19,d_h16_19
0,hero_a,base,0.12081,0.000000,0.16070,0.000000
1,knn_analog,K25,0.12149,0.000673,0.16390,0.003198
2,knn_local,intercept,0.12079,-0.000030,0.16045,-0.000244
3,knn_local,har,0.12072,-0.000093,0.15986,-0.000837
4,knn_local,V,0.12071,-0.000108,0.15988,-0.000816
5,knn_local,all,0.12072,-0.000100,0.15982,-0.000875
6,knn_local,SHUFFLE,0.12109,0.000277,NaN,NaN


PASS - SHUFFLE |d_full|=0.000277 < 5e-4 (leakage-clean); analog hurts (+0.00067), local-ridge best d_full=-0.00011.


**The cluster pipeline behind the `fa_d8c5` base preds (folded).** The Hero-A base QLIKE is recomputed *locally*, but `preds/fa_d8c5.csv` (the d8 global-leftover `pred_adj, y_true, base`) was written by the **CARC run** (`resid_amortized.py`, cluster-only cache). The cascade that produced it — linbest base + per-block d8 global XGB fit inside `preds_chunk` — is shown here; the two kNN stages (`knn_analog.main` / `knn_local.main`, §1) only *consume* these preds. *Values from the cluster run; source shown here.*

<details>
<summary><code>resid_amortized.py  ::  preds_chunk</code></summary>

```python
def preds_chunk(cache, arm, cfg, blk0, blk1):
    """Residualized OOS preds for the rows covered by cadence blocks [blk0, blk1).

    Chunks partition the cadence blocks, so each tree-refit point is fit in exactly ONE
    chunk (no repeated work) and the trial's full preds are the ordered concat of chunks.
    Returns (k0, k1, preds) with k = t - train_win (OOS index)."""
    c = cache
    tw = c["cell"]["train_win"]
    model = c["cell"]["model"]
    n = len(c["Xs"])
    starts = c["starts"]
    k0 = int(starts[blk0]) - tw
    k1 = (int(starts[blk1]) if blk1 < len(starts) else n) - tw
    if arm == "ridge_alone":
        return k0, k1, np.array(c["ridge_oos"][k0:k1], copy=True)
    if arm == "resid_regime":
        # Hero B regime cascade, per cadence block i:
        #   out = base_oos + GLOBAL_tree(survivors) + gated REGIME_tree(leftover, h16-19 only)
        # GLOBAL stage = Hero-A winning XGB (env GLOBAL_CFG) on the resid_subset survivors masks[i],
        # fit on r1 = y_train - base_train (the residualized target). REGIME stage = EBM (env
        # REGIME_CFG) on the LEFTOVER r2 = r1 - global(Xtr), trained on h16-19 TRAIN rows only and
        # predicting ZERO outside h16-19. With REGIME_CFG empty ({} / unset) the regime stage is
        # skipped, so this arm reproduces the single-pass resid_subset number (SANITY invariant --
        # given the resid_subset run uses the SAME xgb cfg as GLOBAL_CFG on an xgb cell).
        if "masks" not in c:
            raise KeyError(
                "resid_regime needs per-block enet survivor masks (run enet_masks first)"
            )
        if c.get("feats") is None:
            raise KeyError(
                "resid_regime needs aligned feats (cell feats.json) for the hour gate"
            )
        out = np.array(c["ridge_oos"][k0:k1], copy=True)
        masks = c["masks"]
        fm = c.get(
            "force_mask"
        )  # FORCE_COLS: union into survivors so the tree/EBM see them
        hr = c["Xs"][:, c["feats"].index("hour")]
        re = c.get(
            "regime_extra"
        )  # REGIME_EXTRA: regime-persistence features injected into the EBM ONLY -- they BYPASS the
        # global enet base (unchanged ridge_oos/coefs) and the global d8 tree (g fits survivors only),
        # entering at the regime stage where there is no global alpha to mis-penalize them. None -> off.
        gcfg = json.loads(os.environ.get("GLOBAL_CFG", "{}"))
        rcfg = json.loads(os.environ.get("REGIME_CFG", "{}"))
        # regime-stage learner: default EBM (interpretable); REGIME_MODEL=xgb -> tuned XGB for PURE
        # POWER (drops the EBM additive/pairwise constraint = the QLIKE ceiling, no interpretability).
        rmodel = os.environ.get("REGIME_MODEL", "ebm")
        regime_full = os.environ.get("REGIME_FULL", "0") not in ("", "0")  # FREE the gate (#1): fit/predict
        # on ALL hours so the learned MoE gate discovers the regime, vs the hand h16-19 pre-gate (default).
        # REGIME_INVERT (#3, structural-starvation test): swap the cascade order to REGIME-then-global, so
        # the soft-routing MoE gets FIRST crack at the un-starved r1 = y - base and d8 only soaks up its
        # leftover. Default (off) = global-then-regime (d8 eats the X-routable structure -> gate starved).
        invert = os.environ.get("REGIME_INVERT", "0") not in ("", "0")
        # REGIME_NOGLOBAL: drop the d8 global stage entirely -> base + regime ONLY (the purest un-starved
        # cascade: additive base + gate-as-sole-regime, no hard router competing). Only honored with invert.
        skip_global = os.environ.get("REGIME_NOGLOBAL", "0") not in ("", "0")
        hr_idx = c["feats"].index("hour")
        mk_g = _tree_factory("xgb", gcfg)
        for i in range(blk0, blk1):
            t_r = int(starts[i])
            cols = masks[i] if fm is None else (masks[i] | fm)
            Xtr = c["Xs"][t_r - tw : t_r]
            t_end = int(starts[i + 1]) if i + 1 < len(starts) else n
            r1 = c["y"][t_r - tw : t_r] - (Xtr @ c["coefs"][i] + c["intercepts"][i])
            if invert:  # REGIME FIRST on the un-starved r1 (gate's first crack), then d8 mops up the leftover
                Xblk = c["Xs"][t_r:t_end][:, cols]
                m_tr = _close_mask(hr[t_r - tw : t_r])
                pe_tr = np.zeros_like(r1)
                if rcfg and (regime_full or int(m_tr.sum()) >= 50):
                    cols_e = cols
                    if regime_full:
                        cols_e = cols.copy()
                        cols_e[hr_idx] = True
                    Xtr_e = Xtr[:, cols_e]
                    Xblk_e = c["Xs"][t_r:t_end][:, cols_e]
                    if re is not None:
                        Xtr_e = np.hstack([Xtr_e, re[t_r - tw : t_r]])
                        Xblk_e = np.hstack([Xblk_e, re[t_r:t_end]])
                    e = _tree_factory(rmodel, rcfg)()
                    if regime_full:
                        e.fit(Xtr_e, r1)
                        out[t_r - tw - k0 : t_end - tw - k0] += e.predict(Xblk_e).ravel()
                        pe_tr = e.predict(Xtr_e).ravel()
                    else:
                        e.fit(Xtr_e[m_tr], r1[m_tr])
                        pe = e.predict(Xblk_e).ravel()
                        pe[~_close_mask(hr[t_r:t_end])] = 0.0
                        out[t_r - tw - k0 : t_end - tw - k0] += pe
                        pe_tr = e.predict(Xtr_e).ravel()
                        pe_tr[~m_tr] = 0.0
                if not skip_global:  # d8 soaks up the regime leftover (off -> base + regime only)
                    g = mk_g()
                    g.fit(Xtr[:, cols], r1 - pe_tr)
                    out[t_r - tw - k0 : t_end - tw - k0] += g.predict(Xblk).ravel()
                continue
            g = mk_g()
            g.fit(Xtr[:, cols], r1)
            Xblk = c["Xs"][t_r:t_end][:, cols]
            out[t_r - tw - k0 : t_end - tw - k0] += g.predict(Xblk).ravel()
            if (
                not rcfg
            ):  # regime stage disabled -> single-pass resid_subset (sanity invariant)
                continue
            m_tr = _close_mask(hr[t_r - tw : t_r])
            if (
                not regime_full and int(m_tr.sum()) < 50
            ):  # too few close/AH train rows -> skip regime this block (predict 0)
                continue
            r2 = r1 - g.predict(Xtr[:, cols]).ravel()
            cols_e = cols
            if regime_full:  # add `hour` so the freed gate can route on the clock itself
                cols_e = cols.copy()
                cols_e[hr_idx] = True
            Xtr_e = Xtr[:, cols_e]
            Xblk_e = c["Xs"][t_r:t_end][:, cols_e]
            if re is not None:  # regime model also sees the persistence extras (global g unaffected above)
                Xtr_e = np.hstack([Xtr_e, re[t_r - tw : t_r]])
                Xblk_e = np.hstack([Xblk_e, re[t_r:t_end]])
            if rmodel == "ebm_mtfm":
                # EBM (+) multi-task-FM ENSEMBLE: the binned-bagged EBM on r2 plus an un-starved multi-task
                # FM (aux head predicts r1, the pre-d8 residual). They are DIVERSE (pre-check corr ~0.34) so
                # the weighted average can beat the EBM alone. ENS_W = weight on the EBM; MT_* = the FM knobs.
                from src.models.regime_moe import MultiTaskFM

                ens_w = float(os.environ.get("ENS_W", "0.4"))
                # un-starve the ANTICIPATION: MT_GATE=ghat -> per-row aux weight gated by |ghat|=|r1-r2|
                # (d8's bite, the taken structure made explicit) — spend the aux where d8 took a big bite.
                gate_kind = os.environ.get("MT_GATE", "uniform")
                aux_hi = float(os.environ.get("MT_AUXHI", "0.6"))
                ghat = r1 - r2  # = g.predict(Xtr[:, cols]); causal, available at train

                def _auxw(sel):
                    if gate_kind != "ghat":
                        return None
                    b = np.abs(ghat[sel])
                    return np.where(b > np.median(b), aux_hi, 0.0).astype(np.float32)

                ebm = _tree_factory("ebm", json.loads(os.environ.get("EBM_CFG", "{}")))()
                mt = MultiTaskFM(
                    rank=int(os.environ.get("MT_RANK", "4")),
                    n_bags=int(os.environ.get("MT_NBAGS", "8")),
                    epochs=int(os.environ.get("MT_EPOCHS", "250")),
                    aux_weight=float(os.environ.get("MT_AUXW", "0.3")),
                    weight_decay=float(os.environ.get("MT_WD", "0.1")),
                )
                if regime_full:
                    ebm.fit(Xtr_e, r2)
                    mt.fit(Xtr_e, r2, r1, aux_w=_auxw(slice(None)))
                    pe = (ens_w * ebm.predict(Xblk_e) + (1 - ens_w) * mt.predict(Xblk_e)).ravel()
                else:
                    ebm.fit(Xtr_e[m_tr], r2[m_tr])
                    mt.fit(Xtr_e[m_tr], r2[m_tr], r1[m_tr], aux_w=_auxw(m_tr))
                    pe = (ens_w * ebm.predict(Xblk_e) + (1 - ens_w) * mt.predict(Xblk_e)).ravel()
                    pe[~_close_mask(hr[t_r:t_end])] = 0.0
            else:
                e = _tree_factory(rmodel, rcfg)()
                if regime_full:  # fit/predict on ALL hours; the learned gate discovers WHERE the regime is
                    e.fit(Xtr_e, r2)
                    pe = e.predict(Xblk_e).ravel()
                else:  # hand pre-gate: fit on h16-19 train rows, predict gated to h16-19
                    e.fit(Xtr_e[m_tr], r2[m_tr])
                    pe = e.predict(Xblk_e).ravel()
                    pe[~_close_mask(hr[t_r:t_end])] = 0.0
            out[t_r - tw - k0 : t_end - tw - k0] += pe
        return k0, k1, out
    raw = arm == "raw_tree"
    out = np.zeros(k1 - k0) if raw else np.array(c["ridge_oos"][k0:k1], copy=True)
    if (
        arm == "resid_subset"
    ):  # tree sees only the block's rolling enet survivors (~120)
        fm = c.get("force_mask")  # FORCE_COLS: union forced cols into the survivor set

        def colsel(i):
            return c["masks"][i] if fm is None else (c["masks"][i] | fm)
    elif (
        arm == "resid_subset_ind"
    ):  # survivors UNION live availability indicators (event channel)
        li = c["live_ind"]

        def colsel(i):
            return c["masks"][i] if li is None else (c["masks"][i] | li)
    elif (
        arm == "resid_subset_nocov"
    ):  # survivors MINUS coverage-artifact indicators (voldemand-type
        cov = c.get(
            "cov_mask"
        )  # availability steps) -> tests if the EBM leans on a data artifact

        def colsel(i):
            return c["masks"][i] if cov is None else (c["masks"][i] & ~cov)
    elif (
        arm == "resid_pruned"
    ):  # tree sees all-but-the-signalless (the 224-indicator prune)
        keep = ~c["prunable"]

        def colsel(i):
            return keep
    else:  # residualized / raw_tree: tree sees all features

        def colsel(i):
            return slice(None)

    mk = _tree_factory(model, cfg)
    for i in range(blk0, blk1):
        t_r = int(starts[i])
        Xtr = c["Xs"][t_r - tw : t_r]
        cols = colsel(i)
        r_train = (
            c["y"][t_r - tw : t_r]
            if raw
            else c["y"][t_r - tw : t_r] - (Xtr @ c["coefs"][i] + c["intercepts"][i])
        )
        tree = mk()
        tree.fit(Xtr[:, cols], r_train)
        t_end = int(starts[i + 1]) if i + 1 < len(starts) else n
        out[t_r - tw - k0 : t_end - tw - k0] += tree.predict(
            c["Xs"][t_r:t_end][:, cols]
        ).ravel()
    return k0, k1, out
```

</details>

## 3 . Interpret

The expected story (now downstream of the verified deltas -- the table above), and why it matters even if it loses:

- **The proper local-linear kNN works** (placebo-clean, best `d_full` slightly negative) but is **dominated by the
  EBM regime** -- kNN is a *validated method, not a deployable lever here*. The naive K-mean analog **hurts**
  (positive `d_full`), which is itself a finding: the close leftover is low-dimensional and noise-dominated, so
  adding unstructured local variance is harmful -- only *structured* local fitting (ridge) extracts the thin signal.
- **kNN vs the MoE gate (ch. 04):** the MoE's soft gate is a *learned-metric* generalization of fixed kNN
  (the gate learns the regime the kNN's hand-metric assumes). If neither beats the EBM, the state framing
  is earned for real and the lever is **information (the auction cross), not more local/relevance modelling**.
- The model class that subsumes both (sequence-attention) is **provably empty here** (log-sig path-lever
  death, ch. 03) -- so the kNN is the right, data-efficient, *legible* probe, not a stepping stone to a
  sequence model.